In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Load the two files from your repository
# Assuming files are in the same directory
file1 = "NEW-DATA-1.T15.txt"
file2 = "NEW-DATA-2.T15.txt"

# Columns as per your description
column_names = [
    "Date", "Time", "Temp_Comedor", "Temp_Habitacion", "Weather_Temp",
    "CO2_Comedor", "CO2_Habitacion", "Hum_Comedor", "Hum_Habitacion",
    "Light_Comedor", "Light_Habitacion", "Precipitation", "Meteo_Crepusculo",
    "Meteo_Wind", "Meteo_Sun_Oest", "Meteo_Sun_Est", "Meteo_Sun_Sud",
    "Meteo_Piranometro", "Entalpic_1", "Entalpic_2", "Entalpic_turbo",
    "Temp_Exterior", "Hum_Exterior", "Day_of_Week"
]

df1 = pd.read_csv(file1, sep='\s+', names=column_names, header=None)
df2 = pd.read_csv(file2, sep='\s+', names=column_names, header=None)
df = pd.concat([df1, df2], ignore_index=True)

# Select features (skip Date and Time)
features_cols = column_names[2:] 
data = df[features_cols].values

# Normalize data (required for stable Int8 quantization)
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data)

# Windowing: Use last 4 time steps (1 hour of data) to predict current Temp_Comedor
def create_windows(dataset, window_size=4):
    X, y = [], []
    for i in range(window_size, len(dataset)):
        X.append(dataset[i-window_size:i, :]) # Previous 4 steps
        y.append(dataset[i, 0])                # Current Temp_Comedor (index 0)
    return np.array(X), np.array(y)

X, y = create_windows(data_scaled)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)